<a href="https://colab.research.google.com/github/alearecuest/anyoneai-exercises-sprint_3/blob/main/02_llm_zero_few_shot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Notebook Overview**

This notebook demonstrates **zero-shot and few-shot inference using Large Language Models (LLMs)**. Instead of fine-tuning, we focus on how prompting alone can enable powerful model behavior.

### **What You Will Learn**

* How to run text classification via prompting (no training)
* How to design *zero-shot*, *one-shot*, and *few-shot* prompts
* How to compare model performance across different prompting strategies
* Practical prompt-engineering tips for classification tasks

### **Key Concepts Covered**

* Zero-shot generalization
* In-context learning
* Prompt design patterns (instruction, question, options, exemplars)
* Differences between model inference and model training


## 1. Import Libraries and Load Model

We'll use the Hugging Face `transformers` and `datasets` libraries.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load Qwen2.5-0.5B-Instruct model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print(f"Model loaded successfully!")

Loading Qwen/Qwen2.5-0.5B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded successfully!


## 2. Helper Function for Generation

In [ ]:
def generate_response(prompt, max_new_tokens=180):
    """
    Generate a response from the model given a prompt.
    """
    # Format as a chat message
    messages = [
        {"role": "user", "content": prompt}
    ]

    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=False,
        top_p=0.9
    )

    # Decode only the generated part
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return response.strip()

## 3. Load a Test Dataset

We'll use the IMDb dataset (same as notebook 1) but with a smaller test set, just to demonstrate zero-shot and few-shot sentiment analysis.

In [ ]:
from datasets import load_dataset

# Load a small subset of IMDb reviews
dataset = load_dataset("imdb")
dataset = dataset["test"].shuffle(seed=42).select(range(50))

print(f"Loaded {len(dataset)} test reviews")
print("\nExample review:")
print(f"Text: {dataset[1]['text']}")
print(f"Label: {'Positive' if dataset[1]['label'] == 1 else 'Negative'}")

Loaded 50 test reviews

Example review:
Text: This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French answer to James Bond). The series was launched in the early 1950's, and spawned at least eight films (none of which was ever released in the U.S.). 'O.S.S.117:Cairo,Nest Of Spies' is a breezy little comedy that should not...repeat NOT, be taken too seriously. Our protagonist finds himself in the middle of a spy chase in Egypt (with Morroco doing stand in for Egypt) to find out about a long lost friend. What follows is the standard James Bond/Inspector Cloussou kind of antics. Although our man is something of an overt xenophobe,sexist,homophobe, it's treated as pure farce (as I said, don't take it too seriously). Although there is a bit of rough language & cartoon violence, it's basically okay for older kids (ages 12 & up). As previously stated in the subject line, just sit back,pass the popcorn & just enjoy.
Label: Positive


In [ ]:
# See the positive and negative class balance:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 50
})

## 4. Zero-Shot Sentiment Analysis

Let's try sentiment classification without providing any examples - just a direct instruction.

In [ ]:
print("=== Zero-Shot Sentiment Analysis ===\n")

# Test on first 5 examples
correct = 0
for i in range(5):
    review = dataset[i]['text']
    true_label = "positive" if dataset[i]['label'] == 1 else "negative"

    prompt = f"Classify the sentiment of this movie review as either 'positive' or 'negative'.\n\nReview: {review}\n\nSentiment:"

    prediction = generate_response(prompt, max_new_tokens=100).lower()

    # Check if prediction contains the correct sentiment
    is_correct = true_label in prediction
    if is_correct:
        correct += 1

    print(f"Review {i+1}:")
    print(f"Text: {review[:100]}...")
    print(f"True: {true_label} | Predicted: {prediction} | {'✓' if is_correct else '✗'}")
    print()

print(f"Zero-Shot Accuracy: {correct}/5 = {correct/5*100:.1f}%")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Zero-Shot Sentiment Analysis ===

Review 1:
Text: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining Kin...
True: positive | Predicted: the sentiment of this movie review is clearly positive. the reviewer expresses strong appreciation for the film, praising its performances, themes, and overall impact. they describe the movie as "wonderfully subtle and compassionate," and use phrases like "eye-opener" and "relief" to emphasize their positive feelings towards the film. the reviewer even goes so far as to say they will continue watching it multiple times, indicating their high level of enjoyment and satisfaction with the movie. overall, the tone and content | ✓

Review 2:
Text: This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French a...
True: positive | Predicted: the sentiment of this movie review can be classified as **neutral**. while the reviewer does mention some positive aspects such as "b

## 5. Few-Shot Sentiment Analysis

Now let's provide a few examples to guide the model's responses.

In [ ]:
print("=== Few-Shot Sentiment Analysis ===\n")

# Create a few-shot prompt with examples - use fewer, clearer examples
few_shot_template = """Classify the sentiment as 'positive' or 'negative'.

E.g.
Review: This film was absolutely brilliant! The acting was superb and the story kept me engaged.
Sentiment: positive

Review: What a waste of time. Poor acting and boring from start to finish.
Sentiment: negative

Now classify the sentiment into only positive or negative of the following review:
Review: {review}
Sentiment:"""

# Test on the same 5 examples
correct = 0
for i in range(5):
    review = dataset[i]['text']
    true_label = "positive" if dataset[i]['label'] == 1 else "negative"

    prompt = few_shot_template.format(review=review)
    prediction = generate_response(prompt, max_new_tokens=100).lower()

    is_correct = true_label in prediction
    if is_correct:
        correct += 1

    print(f"Review {i+1}:")
    print(f"Text: {review[:100]}...")
    print(f"True: {true_label} | Predicted: {prediction} | {'✓' if is_correct else '✗'}")
    print()

print(f"Few-Shot Accuracy: {correct}/5 = {correct/5*100:.1f}%")

=== Few-Shot Sentiment Analysis ===

Review 1:
Text: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining Kin...
True: positive | Predicted: positive | ✓

Review 2:
Text: This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French a...
True: positive | Predicted: positive | ✓

Review 3:
Text: This movie was so frustrating. Everything seemed energetic and I was totally prepared to have a good...
True: negative | Predicted: the sentiment in this review is clearly **negative**. the reviewer expresses strong dissatisfaction with various aspects of the movie, including the acting, plot, pacing, and overall quality. they use phrases like "absolutely brilliant", "boring", "wasted time", and "horrible" to describe their experience, indicating a highly negative sentiment towards the film. | ✓

Review 4:
Text: I was truly and wonderfully surprised at "O' Brother, Where Art Thou?" The video store was out of al...
T

## 6. Comparison: Zero-Shot vs Few-Shot

Let's run a larger comparison to see which approach works better.

In [ ]:
print("=== Running Comparison on 20 Reviews ===\n")

zero_shot_correct = 0
few_shot_correct = 0

for i in range(20):
    review = dataset[i]['text']
    true_label = "positive" if dataset[i]['label'] == 1 else "negative"

    # Zero-shot
    zero_prompt = f"Classify the sentiment of this movie review as either 'positive' or 'negative'.\n\nReview: {review}\n\nSentiment:"
    zero_pred = generate_response(zero_prompt, max_new_tokens=100).lower()
    if true_label in zero_pred:
        zero_shot_correct += 1

    # Few-shot
    few_shot_prompt = few_shot_template.format(review=review)
    few_pred = generate_response(few_shot_prompt, max_new_tokens=100).lower()
    if true_label in few_pred:
        few_shot_correct += 1

    print(f"Review {i+1}: True={true_label} | Zero-shot={'✓' if true_label in zero_pred else '✗'} | Few-shot={'✓' if true_label in few_pred else '✗'}")

print(f"\n{'='*50}")
print(f"Zero-Shot Accuracy: {zero_shot_correct}/20 = {zero_shot_correct/20*100:.1f}%")
print(f"Few-Shot Accuracy:  {few_shot_correct}/20 = {few_shot_correct/20*100:.1f}%")

=== Running Comparison on 20 Reviews ===

Review 1: True=positive | Zero-shot=✓ | Few-shot=✓
Review 2: True=positive | Zero-shot=✓ | Few-shot=✓
Review 3: True=negative | Zero-shot=✓ | Few-shot=✓
Review 4: True=positive | Zero-shot=✓ | Few-shot=✓
Review 5: True=negative | Zero-shot=✓ | Few-shot=✓
Review 6: True=positive | Zero-shot=✓ | Few-shot=✓
Review 7: True=positive | Zero-shot=✓ | Few-shot=✓
Review 8: True=negative | Zero-shot=✓ | Few-shot=✓
Review 9: True=negative | Zero-shot=✓ | Few-shot=✓
Review 10: True=positive | Zero-shot=✓ | Few-shot=✓
Review 11: True=positive | Zero-shot=✓ | Few-shot=✓
Review 12: True=negative | Zero-shot=✓ | Few-shot=✓
Review 13: True=negative | Zero-shot=✓ | Few-shot=✓
Review 14: True=negative | Zero-shot=✓ | Few-shot=✓
Review 15: True=positive | Zero-shot=✓ | Few-shot=✓
Review 16: True=positive | Zero-shot=✓ | Few-shot=✓
Review 17: True=negative | Zero-shot=✓ | Few-shot=✓
Review 18: True=negative | Zero-shot=✓ | Few-shot=✗
Review 19: True=positive | Zero